# SAM2 Fine-Tuning for IKEA Manual Part Instance Segmentation

This notebook fine-tunes SAM2.1 on IKEA assembly manual step images to segment individual parts.

**Prerequisites:**
1. Run `python -m part_segmentation.data_preparation` locally to create `data/sam2_dataset.zip`
2. Upload `sam2_dataset.zip` to Google Drive root
3. Upload `checkpoints/sam2.1_b.pt` to Google Drive root

**Runtime:** Select **A100 GPU** in Colab Pro.

## Cell 1 — Environment Setup

In [ ]:
!pip install -q sam2 pycocotools albumentations

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pycocotools.mask as mask_util

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Cell 2 — Unpack Dataset

In [ ]:
# Adjust these paths if you uploaded to a different Drive folder
DRIVE_DATASET = "/content/drive/MyDrive/sam2_dataset.zip"
DRIVE_CHECKPOINT = "/content/drive/MyDrive/sam2.1_b.pt"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/sam2_checkpoints"  # saved models go here

DATASET_DIR = "/content/sam2_dataset"
TRAIN_JSON = os.path.join(DATASET_DIR, "sam2_dataset", "train", "annotations.json")
TEST_JSON = os.path.join(DATASET_DIR, "sam2_dataset", "test", "annotations.json")
TRAIN_IMAGES = os.path.join(DATASET_DIR, "sam2_dataset", "train", "images")
TEST_IMAGES = os.path.join(DATASET_DIR, "sam2_dataset", "test", "images")

# Unpack
if not os.path.exists(TRAIN_JSON):
    !unzip -q {DRIVE_DATASET} -d {DATASET_DIR}
    print("Dataset unpacked.")
else:
    print("Dataset already unpacked.")

# Verify
with open(TRAIN_JSON) as f:
    train_coco = json.load(f)
with open(TEST_JSON) as f:
    test_coco = json.load(f)

print(f"Train: {len(train_coco['images'])} images, {len(train_coco['annotations'])} instances")
print(f"Test:  {len(test_coco['images'])} images, {len(test_coco['annotations'])} instances")

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

## Cell 3 — Visualize Data (Sanity Check)

In [ ]:
def visualize_sample(coco_data, images_dir, idx=0):
    """Overlay instance masks on a step crop image."""
    img_info = coco_data["images"][idx]
    img_path = os.path.join(images_dir, img_info["file_name"])
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Get annotations for this image
    anns = [a for a in coco_data["annotations"] if a["image_id"] == img_info["id"]]

    overlay = image.copy()
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(anns), 1)))

    for i, ann in enumerate(anns):
        seg = ann["segmentation"]
        rle = {"size": seg["size"], "counts": seg["counts"].encode("utf-8")}
        mask = mask_util.decode(rle)
        color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
        overlay[mask > 0] = (overlay[mask > 0] * 0.5 + color * 0.5).astype(np.uint8)

        # Draw bbox
        x, y, w, h = ann["bbox"]
        cv2.rectangle(overlay, (x, y), (x+w, y+h), color.tolist(), 2)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(image)
    axes[0].set_title(f"Original: {img_info['file_name']}")
    axes[0].axis("off")
    axes[1].imshow(overlay)
    axes[1].set_title(f"{len(anns)} instance masks")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Show 8 random samples from training set
indices = random.sample(range(len(train_coco["images"])), min(8, len(train_coco["images"])))
for idx in indices:
    visualize_sample(train_coco, TRAIN_IMAGES, idx)

## Cell 4 — Configuration

In [ ]:
# ============================================================
# TRAINING CONFIGURATION — edit these as needed
# ============================================================

# Model
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_b+.yaml"  # base_plus for A100
CHECKPOINT = DRIVE_CHECKPOINT

# Training mode: "full" (recommended) or "decoder"
MODE = "full"

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 4           # Increase to 8 if VRAM allows
BACKBONE_LR = 1e-6       # Only used in "full" mode
DECODER_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
INPUT_SIZE = 1024
BOX_JITTER_PX = 10

# Loss weights
FOCAL_WEIGHT = 20.0
DICE_WEIGHT = 1.0

# Checkpointing
SAVE_EVERY = 5            # Save to Drive every N epochs

# Resume from a previous checkpoint (set to None for fresh start)
RESUME_FROM = None        # e.g., "/content/drive/MyDrive/sam2_checkpoints/sam2_ikea_epoch_10.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Config: MODE={MODE}, EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")

## Cell 5 — Dataset, Model & Training Loop

In [ ]:
# ---- Dataset ----

class SAM2FineTuneDataset(Dataset):
    """One sample = (image, single_instance_mask, box_prompt)."""

    def __init__(self, annotations_json, images_dir, input_size=1024,
                 box_jitter_px=10, augment=False):
        with open(annotations_json) as f:
            coco = json.load(f)
        self.images_dir = images_dir
        self.input_size = input_size
        self.box_jitter_px = box_jitter_px
        self.augment = augment
        self.images = {img["id"]: img for img in coco["images"]}
        self.samples = [a for a in coco["annotations"] if a["area"] >= 10]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ann = self.samples[idx]
        img_info = self.images[ann["image_id"]]

        # Load image
        img_path = os.path.join(self.images_dir, img_info["file_name"])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]

        # Decode mask
        seg = ann["segmentation"]
        rle = {"size": seg["size"],
               "counts": seg["counts"].encode("utf-8") if isinstance(seg["counts"], str) else seg["counts"]}
        mask = mask_util.decode(rle)

        # Augmentation
        if self.augment:
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                mask = np.fliplr(mask).copy()
            if random.random() < 0.3:
                alpha = random.uniform(0.8, 1.2)
                beta = random.randint(-20, 20)
                image = np.clip(image * alpha + beta, 0, 255).astype(np.uint8)

        # Box prompt from mask with jitter
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        if not rows.any():
            y1, y2, x1, x2 = 0, 1, 0, 1
        else:
            y1, y2 = np.where(rows)[0][[0, -1]]
            x1, x2 = np.where(cols)[0][[0, -1]]

        j = self.box_jitter_px
        if j > 0:
            x1 = max(0, x1 + random.randint(-j, j))
            y1 = max(0, y1 + random.randint(-j, j))
            x2 = min(orig_w, x2 + random.randint(-j, j))
            y2 = min(orig_h, y2 + random.randint(-j, j))

        # Resize
        sx = self.input_size / orig_w
        sy = self.input_size / orig_h
        image = cv2.resize(image, (self.input_size, self.input_size))
        mask = cv2.resize(mask, (self.input_size, self.input_size),
                          interpolation=cv2.INTER_NEAREST)

        box = np.array([x1 * sx, y1 * sy, x2 * sx, y2 * sy], dtype=np.float32)

        image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)
        box_t = torch.from_numpy(box)

        return {"image": image_t, "mask": mask_t, "box": box_t}


train_ds = SAM2FineTuneDataset(TRAIN_JSON, TRAIN_IMAGES, INPUT_SIZE, BOX_JITTER_PX, augment=True)
val_ds = SAM2FineTuneDataset(TEST_JSON, TEST_IMAGES, INPUT_SIZE, box_jitter_px=0, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# ---- Model ----

from sam2.build_sam import build_sam2

sam2_model = build_sam2(MODEL_CFG, CHECKPOINT)
sam2_model = sam2_model.to(DEVICE)

# Freeze/unfreeze based on mode
if MODE == "decoder":
    # Freeze image encoder
    for param in sam2_model.image_encoder.parameters():
        param.requires_grad = False
    print("Mode: decoder-only (image encoder frozen)")
elif MODE == "full":
    print("Mode: full fine-tune (all parameters trainable)")

# Count parameters
total_params = sum(p.numel() for p in sam2_model.parameters())
trainable_params = sum(p.numel() for p in sam2_model.parameters() if p.requires_grad)
print(f"Parameters: {total_params/1e6:.1f}M total, {trainable_params/1e6:.1f}M trainable")

In [ ]:
# ---- Loss Functions ----

def sigmoid_focal_loss(pred, target, alpha=0.25, gamma=2.0):
    """Sigmoid focal loss for binary segmentation."""
    prob = torch.sigmoid(pred)
    ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
    p_t = prob * target + (1 - prob) * (1 - target)
    focal_weight = (alpha * target + (1 - alpha) * (1 - target)) * (1 - p_t) ** gamma
    return (focal_weight * ce_loss).mean()


def dice_loss(pred, target, smooth=1.0):
    """Dice loss for binary segmentation."""
    pred = torch.sigmoid(pred)
    pred_flat = pred.flatten(1)
    target_flat = target.flatten(1)
    intersection = (pred_flat * target_flat).sum(1)
    return 1 - ((2.0 * intersection + smooth) / (pred_flat.sum(1) + target_flat.sum(1) + smooth)).mean()


def compute_iou(pred_mask, gt_mask):
    """Compute IoU between predicted and ground-truth binary masks."""
    pred_bin = (torch.sigmoid(pred_mask) > 0.5).float()
    intersection = (pred_bin * gt_mask).sum(dim=(1, 2, 3))
    union = ((pred_bin + gt_mask) > 0).float().sum(dim=(1, 2, 3))
    return (intersection / (union + 1e-6))

In [ ]:
# ---- Optimizer & Scheduler ----

if MODE == "full":
    # Differential learning rates
    encoder_params = list(sam2_model.image_encoder.parameters())
    other_params = [p for n, p in sam2_model.named_parameters()
                    if "image_encoder" not in n and p.requires_grad]
    param_groups = [
        {"params": encoder_params, "lr": BACKBONE_LR},
        {"params": other_params, "lr": DECODER_LR},
    ]
else:
    param_groups = [{"params": [p for p in sam2_model.parameters() if p.requires_grad],
                     "lr": DECODER_LR}]

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_loader)
warmup_steps = min(WARMUP_STEPS, total_steps // 5)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda")

print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")

In [ ]:
# ---- Forward pass helper ----

def forward_sam2(model, images, boxes):
    """Run SAM2 forward pass with box prompts.

    Args:
        model: SAM2 model.
        images: (B, 3, H, W) tensor.
        boxes: (B, 4) tensor [x1, y1, x2, y2].

    Returns:
        pred_masks: (B, 1, H, W) logits.
        iou_pred: (B, 1) predicted IoU scores.
    """
    # Encode images
    backbone_out = model.forward_image(images)
    _, vision_feats, _, _ = model._prepare_backbone_features(backbone_out)

    # Add no-memory context for image-only mode
    # (SAM2 expects memory context even for single images)
    B = images.shape[0]

    # Get high-res feature maps
    if hasattr(model, 'sam_prompt_encoder') and hasattr(model, 'sam_mask_decoder'):
        # SAM2 API: use sam_prompt_encoder and sam_mask_decoder
        sparse_embeddings, dense_embeddings = model.sam_prompt_encoder(
            points=None,
            boxes=boxes,
            masks=None,
        )

        # Get the high-res features for mask decoder
        # vision_feats is a list of multi-scale features
        high_res_feats = [vision_feats[0], vision_feats[1]]
        image_embeddings = vision_feats[-1]

        # Reshape if needed
        if image_embeddings.dim() == 3:
            # (B, N, C) -> (B, C, H, W)
            hw = int(image_embeddings.shape[1] ** 0.5)
            image_embeddings = image_embeddings.transpose(1, 2).reshape(B, -1, hw, hw)

        for i in range(len(high_res_feats)):
            if high_res_feats[i].dim() == 3:
                hw = int(high_res_feats[i].shape[1] ** 0.5)
                high_res_feats[i] = high_res_feats[i].transpose(1, 2).reshape(B, -1, hw, hw)

        low_res_masks, iou_pred, _, _ = model.sam_mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False,
            repeat_image=False,
            high_res_features=high_res_feats,
        )
    else:
        raise RuntimeError("Could not find SAM2 encoder/decoder attributes. "
                           "Check SAM2 version compatibility.")

    # Upscale masks to input resolution
    pred_masks = F.interpolate(
        low_res_masks, size=(INPUT_SIZE, INPUT_SIZE),
        mode="bilinear", align_corners=False
    )

    return pred_masks, iou_pred

In [ ]:
# ---- Training Loop ----

# Resume if specified
start_epoch = 0
best_val_iou = 0.0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    sam2_model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_iou = ckpt.get("best_val_iou", 0.0)
    print(f"Resumed from epoch {start_epoch}, best_val_iou={best_val_iou:.4f}")

# Training history
history = {"train_loss": [], "val_iou": [], "val_loss": [], "lr": []}

print(f"\nStarting training: epochs {start_epoch}-{EPOCHS-1}")
print("=" * 70)

for epoch in range(start_epoch, EPOCHS):
    # ---- Train ----
    sam2_model.train()
    epoch_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        gt_masks = batch["mask"].to(DEVICE)
        boxes = batch["box"].to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            pred_masks, iou_pred = forward_sam2(sam2_model, images, boxes)

            loss_focal = sigmoid_focal_loss(pred_masks, gt_masks)
            loss_dice = dice_loss(pred_masks, gt_masks)
            gt_iou = compute_iou(pred_masks.detach(), gt_masks)
            loss_iou = F.mse_loss(iou_pred.squeeze(-1), gt_iou)

            loss = FOCAL_WEIGHT * loss_focal + DICE_WEIGHT * loss_dice + loss_iou

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sam2_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_train_loss = epoch_loss / max(num_batches, 1)
    current_lr = optimizer.param_groups[-1]["lr"]

    # ---- Validate ----
    sam2_model.eval()
    val_loss_sum = 0.0
    val_iou_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(DEVICE)
            gt_masks = batch["mask"].to(DEVICE)
            boxes = batch["box"].to(DEVICE)

            with torch.amp.autocast("cuda"):
                pred_masks, iou_pred = forward_sam2(sam2_model, images, boxes)
                loss_focal = sigmoid_focal_loss(pred_masks, gt_masks)
                loss_dice = dice_loss(pred_masks, gt_masks)
                val_loss = FOCAL_WEIGHT * loss_focal + DICE_WEIGHT * loss_dice

            batch_iou = compute_iou(pred_masks, gt_masks)
            val_loss_sum += val_loss.item() * images.shape[0]
            val_iou_sum += batch_iou.sum().item()
            val_count += images.shape[0]

    avg_val_loss = val_loss_sum / max(val_count, 1)
    avg_val_iou = val_iou_sum / max(val_count, 1)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_iou"].append(avg_val_iou)
    history["lr"].append(current_lr)

    # Print progress
    improved = ""
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        improved = " *BEST*"
        # Save best model
        torch.save({
            "model_state_dict": sam2_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_iou": best_val_iou,
        }, os.path.join(DRIVE_OUTPUT_DIR, "sam2_ikea_best.pt"))

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"train_loss={avg_train_loss:.4f} | "
          f"val_loss={avg_val_loss:.4f} | "
          f"val_iou={avg_val_iou:.4f}{improved} | "
          f"lr={current_lr:.2e}")

    # Periodic checkpoint to Drive
    if (epoch + 1) % SAVE_EVERY == 0:
        ckpt_path = os.path.join(DRIVE_OUTPUT_DIR, f"sam2_ikea_epoch_{epoch}.pt")
        torch.save({
            "model_state_dict": sam2_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_iou": best_val_iou,
        }, ckpt_path)
        print(f"  -> Checkpoint saved to {ckpt_path}")

print("\n" + "=" * 70)
print(f"Training complete. Best val IoU: {best_val_iou:.4f}")
print(f"Best model saved to: {os.path.join(DRIVE_OUTPUT_DIR, 'sam2_ikea_best.pt')}")

## Cell 6 — Evaluation

In [ ]:
# ---- Plot Training History ----

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["val_iou"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].set_title(f"Validation IoU (best={best_val_iou:.4f})")

axes[2].plot(history["lr"])
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("LR")
axes[2].set_title("Learning Rate")

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

In [ ]:
# ---- Quantitative Evaluation ----

# Load best model
best_ckpt = torch.load(os.path.join(DRIVE_OUTPUT_DIR, "sam2_ikea_best.pt"),
                        map_location=DEVICE)
sam2_model.load_state_dict(best_ckpt["model_state_dict"])
sam2_model.eval()

# Run on test set
all_ious = []
per_category_ious = {}  # category -> list of IoUs

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(DEVICE)
        gt_masks = batch["mask"].to(DEVICE)
        boxes = batch["box"].to(DEVICE)

        with torch.amp.autocast("cuda"):
            pred_masks, _ = forward_sam2(sam2_model, images, boxes)

        batch_ious = compute_iou(pred_masks, gt_masks)
        all_ious.extend(batch_ious.cpu().numpy())

all_ious = np.array(all_ious)
print(f"\nTest Set Results (fine-tuned SAM2):")
print(f"  Mean IoU: {all_ious.mean():.4f}")
print(f"  Median IoU: {np.median(all_ious):.4f}")
print(f"  IoU > 0.5: {(all_ious > 0.5).mean() * 100:.1f}%")
print(f"  IoU > 0.75: {(all_ious > 0.75).mean() * 100:.1f}%")

# IoU distribution
plt.figure(figsize=(8, 5))
plt.hist(all_ious, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title(f"IoU Distribution (mean={all_ious.mean():.3f})")
plt.axvline(all_ious.mean(), color="red", linestyle="--", label=f"Mean={all_ious.mean():.3f}")
plt.legend()
plt.savefig(os.path.join(DRIVE_OUTPUT_DIR, "iou_distribution.png"), dpi=150)
plt.show()

In [ ]:
# ---- Visual Comparison: Fine-tuned vs Ground Truth ----

sam2_model.eval()

# Show 6 random test samples
indices = random.sample(range(len(val_ds)), min(6, len(val_ds)))

fig, axes = plt.subplots(len(indices), 3, figsize=(18, 6 * len(indices)))
if len(indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    sample = val_ds[idx]
    image = sample["image"].unsqueeze(0).to(DEVICE)
    gt_mask = sample["mask"].numpy().squeeze()
    box = sample["box"].unsqueeze(0).to(DEVICE)

    with torch.no_grad(), torch.amp.autocast("cuda"):
        pred_mask, _ = forward_sam2(sam2_model, image, box)
    pred_mask = (torch.sigmoid(pred_mask) > 0.5).cpu().numpy().squeeze()

    img_np = sample["image"].permute(1, 2, 0).numpy()

    # Original
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title("Input")
    axes[row, 0].axis("off")

    # Ground truth
    gt_overlay = img_np.copy()
    gt_overlay[gt_mask > 0] = gt_overlay[gt_mask > 0] * 0.5 + np.array([0, 1, 0]) * 0.5
    axes[row, 1].imshow(gt_overlay)
    axes[row, 1].set_title("Ground Truth")
    axes[row, 1].axis("off")

    # Prediction
    pred_overlay = img_np.copy()
    pred_overlay[pred_mask > 0] = pred_overlay[pred_mask > 0] * 0.5 + np.array([1, 0, 0]) * 0.5
    iou = (gt_mask * pred_mask).sum() / ((gt_mask + pred_mask) > 0).sum()
    axes[row, 2].imshow(pred_overlay)
    axes[row, 2].set_title(f"Prediction (IoU={iou:.3f})")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT_DIR, "visual_comparison.png"), dpi=150)
plt.show()

## Cell 7 — Export

The best model is already saved to Google Drive at:
```
MyDrive/sam2_checkpoints/sam2_ikea_best.pt
```

To use locally:
1. Download `sam2_ikea_best.pt` from Drive to `VLM_assembly_plan_gen/checkpoints/`
2. Run: `python -m part_segmentation.predict --checkpoint checkpoints/sam2_ikea_best.pt --output_dir data/sam2_mask/`
3. Run pipeline: `python inference/run.py --model gpt-4o --prompt_type numbered --use_sam2`

In [ ]:
# Summary
print("=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"Mode:           {MODE}")
print(f"Model:          {MODEL_CFG}")
print(f"Epochs:         {EPOCHS}")
print(f"Best val IoU:   {best_val_iou:.4f}")
print(f"Checkpoint:     {os.path.join(DRIVE_OUTPUT_DIR, 'sam2_ikea_best.pt')}")
print(f"\nTraining curves: {os.path.join(DRIVE_OUTPUT_DIR, 'training_curves.png')}")
print(f"IoU histogram:   {os.path.join(DRIVE_OUTPUT_DIR, 'iou_distribution.png')}")
print(f"Visual compare:  {os.path.join(DRIVE_OUTPUT_DIR, 'visual_comparison.png')}")